# Why OLS Fails: Regularisation Bias and Post-Selection Inference

**Goal:** See why Lasso + OLS produces biased causal estimates and invalid confidence intervals.

**Time:** 15 minutes

In this notebook, you will run Lasso to select variables, then OLS on the selection,
and watch the confidence intervals fail. Then you will compare post-Lasso OLS,
double selection, and a DML preview.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## The Setup: High-Dimensional Confounding

We simulate a carbon tax scenario with 200 controls (macro indicators, fuel prices, weather, etc.).
Only a handful truly matter, but some confounders of D are weak predictors of Y.

In [ ]:
n = 1000
p = 200
true_effect = 1.5

X = np.random.randn(n, p)

# Treatment (carbon tax increase) correlated with X0, X1, X3, X7
D = (0.3 * X[:, 0] + 0.2 * X[:, 1] + 0.15 * X[:, 3]
     + 0.1 * X[:, 7] + np.random.randn(n) * 0.5)

# Outcome (power generation cost) depends on D and some different controls
Y = (true_effect * D + 0.8 * X[:, 0] + 0.5 * X[:, 2]
     + 0.3 * X[:, 3] + 0.2 * X[:, 5] + np.random.randn(n))

print(f'n={n}, p={p}, true effect={true_effect}')
print(f'Confounders of D: X0, X1, X3, X7')
print(f'Predictors of Y:  X0, X2, X3, X5')
print(f'Overlap: X0, X3 (confound D AND predict Y)')
print(f'X1, X7 confound D but are weak predictors of Y')

## Step 1: Lasso Variable Selection

Run Lasso on Y ~ X to find the best predictors. Check which confounders of D are retained.

In [ ]:
lasso = LassoCV(cv=5, random_state=42).fit(X, Y)
selected = np.where(np.abs(lasso.coef_) > 0)[0]

print(f'Lasso selected {len(selected)} of {p} variables')
print(f'Selected indices: {selected}')
print()
print('Confounder check:')
for idx in [0, 1, 3, 7]:
    status = 'KEPT' if idx in selected else 'DROPPED'
    print(f'  X{idx} (confounds D): {status}')
print()
print('Predictor check:')
for idx in [0, 2, 3, 5]:
    status = 'KEPT' if idx in selected else 'DROPPED'
    print(f'  X{idx} (predicts Y): {status}')

## Step 2: Post-Selection OLS

Run OLS on the Lasso-selected variables. Compare to the true effect.

In [ ]:
X_sel = X[:, selected]
DX_sel = np.column_stack([D, X_sel])
post_lasso = sm.OLS(Y, sm.add_constant(DX_sel)).fit()

print(f'True effect:          {true_effect:.2f}')
print(f'Post-Lasso OLS:       {post_lasso.params[1]:.2f}')
print(f'SE (ignores selection):{post_lasso.bse[1]:.3f}')
ci = post_lasso.conf_int()[1]
print(f'95% CI:               [{ci[0]:.3f}, {ci[1]:.3f}]')
covers = ci[0] <= true_effect <= ci[1]
print(f'True effect in CI:    {covers}')

## Monte Carlo: Coverage Check

One realisation is not enough. Run 200 simulations to check how often the 95% CI
actually contains the true effect.

In [ ]:
n_sims = 200
results = {'post_lasso': [], 'double_sel': [], 'dml': []}

for sim in range(n_sims):
    rng = np.random.RandomState(sim)
    X_s = rng.randn(n, p)
    D_s = 0.3 * X_s[:, 0] + 0.2 * X_s[:, 1] + 0.15 * X_s[:, 3] + rng.randn(n) * 0.5
    Y_s = true_effect * D_s + 0.8 * X_s[:, 0] + 0.5 * X_s[:, 2] + 0.3 * X_s[:, 3] + rng.randn(n)

    # Post-Lasso OLS
    las = LassoCV(cv=3, random_state=sim).fit(X_s, Y_s)
    sel_y = set(np.where(np.abs(las.coef_) > 0)[0])
    if len(sel_y) == 0:
        sel_y = {0}
    X_sel_s = X_s[:, sorted(sel_y)]
    DX_sel_s = np.column_stack([D_s, X_sel_s])
    m1 = sm.OLS(Y_s, sm.add_constant(DX_sel_s)).fit()
    ci1 = m1.conf_int()[1]
    results['post_lasso'].append(ci1[0] <= true_effect <= ci1[1])

    # Double Selection
    las_d = LassoCV(cv=3, random_state=sim).fit(X_s, D_s)
    sel_d = set(np.where(np.abs(las_d.coef_) > 0)[0])
    sel_union = sorted(sel_y | sel_d)
    if len(sel_union) == 0:
        sel_union = [0]
    DX_u = np.column_stack([D_s, X_s[:, sel_union]])
    m2 = sm.OLS(Y_s, sm.add_constant(DX_u)).fit()
    ci2 = m2.conf_int()[1]
    results['double_sel'].append(ci2[0] <= true_effect <= ci2[1])

    # DML preview
    rf = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=sim)
    Y_hat = cross_val_predict(rf, X_s, Y_s, cv=3)
    D_hat = cross_val_predict(rf, X_s, D_s, cv=3)
    rY = Y_s - Y_hat
    rD = D_s - D_hat
    m3 = sm.OLS(rY, sm.add_constant(rD)).fit()
    ci3 = m3.conf_int()[1]
    results['dml'].append(ci3[0] <= true_effect <= ci3[1])

print(f'Coverage results ({n_sims} simulations):')
print(f'  Post-Lasso OLS:  {np.mean(results["post_lasso"]):.1%}')
print(f'  Double Selection: {np.mean(results["double_sel"]):.1%}')
print(f'  DML (RF preview): {np.mean(results["dml"]):.1%}')
print(f'  Nominal:          95.0%')

## Visualise: Coverage Comparison

Bar chart showing actual vs nominal coverage for each method.

In [ ]:
methods = ['Post-Lasso\nOLS', 'Double\nSelection', 'DML\n(RF preview)']
coverages = [np.mean(results['post_lasso']),
             np.mean(results['double_sel']),
             np.mean(results['dml'])]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['crimson' if c < 0.90 else 'orange' if c < 0.93 else 'forestgreen'
          for c in coverages]
bars = ax.bar(methods, coverages, color=colors, edgecolor='black', linewidth=1.2)
ax.axhline(y=0.95, color='black', linestyle='--', linewidth=2, label='Nominal 95%')
ax.set_ylabel('Actual Coverage Rate', fontsize=12)
ax.set_title('95% CI Coverage: Post-Selection vs DML', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=11)

for bar, cov in zip(bars, coverages):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{cov:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('Post-Lasso OLS dramatically undershoot nominal coverage.')
print('DML achieves near-nominal coverage even with 200 controls.')

## Summary

**What you learned:**
1. Lasso selects variables to predict Y, not to remove confounding from D
2. Post-Lasso OLS standard errors ignore selection uncertainty, giving invalid CIs
3. Actual coverage of "95% CIs" from post-Lasso OLS can be as low as 60-80%
4. Double selection (Lasso on both Y and D) improves but does not fully fix the problem
5. DML achieves near-nominal coverage by using flexible ML with orthogonal scores

**What is next:**
- **Module 02:** The orthogonalisation trick -- how to residualise with ML properly
- **Module 03:** Neyman orthogonal scores -- the formal guarantee behind DML

**Key takeaway:** Variable selection and causal inference have different objectives.
Prediction-oriented tools (Lasso) solve the wrong problem for causal effects.

In [ ]:
learning_objectives(["Lasso selects variables to predict Y, not to remove confounding from D", "Post-Lasso OLS standard errors ignore selection uncertainty, giving invalid CIs", "Actual coverage of "95% CIs" from post-Lasso OLS can be as low as 60-80%", "Double selection (Lasso on both Y and D) improves but does not fully fix the problem", "DML achieves near-nominal coverage by using flexible ML with orthogonal scores"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])